In [2]:
!pip install ccxt

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   --------------- ------------------------ 2.6/6.6 MB 12.6 MB/s eta 0:00:01
   ---------------------- ----------------- 3.7/6.6 MB 9.5 MB/s eta 0:00:01
   ---------------------------------- ----- 5.8/6.6 MB 9.5 MB/s eta 0:00:01
   ---------------------------------------  6.6/6.6 MB 8.0 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 7.8 MB/s  0:00:00
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 12.0 MB/s  0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   --------------------------- ------------ 2.4/3.5 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 11.3 MB/s  0:00:00

   ------ ---------------------------------  2/12 [frozenlist]
   ---------- -----------------------------  3/12 [coincurve]
   ---------------- -----------------------  5/12 [yarl]
   ----


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import ccxt
import pandas as pd

exchange = ccxt.binance()

symbol = 'BTC/USDT'
timeframe = '15m'

ohlcv = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=1000)

df = pd.DataFrame(ohlcv, columns=[
    'timestamp', 'open', 'high', 'low', 'close', 'volume'
])

df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

# Condition: Open == High
df['open_eq_high'] = df['open'] == df['high']

# Filter
signal_df = df[df['open_eq_high']]

print(signal_df[['timestamp', 'open', 'high', 'low', 'close']])
print("Total signals:", len(signal_df))

              timestamp      open      high       low     close
16  2026-03-12 03:45:00  69561.36  69561.36  69285.71  69446.61
19  2026-03-12 04:30:00  69485.49  69485.49  69300.00  69384.55
25  2026-03-12 06:00:00  69480.54  69480.54  69315.63  69453.84
38  2026-03-12 09:15:00  69790.15  69790.15  69580.00  69678.29
83  2026-03-12 20:30:00  70307.03  70307.03  70128.41  70130.34
193 2026-03-14 00:00:00  70930.01  70930.01  70744.01  70889.38
217 2026-03-14 06:00:00  71013.75  71013.75  70892.33  70925.68
218 2026-03-14 06:15:00  70925.69  70925.69  70700.00  70711.90
222 2026-03-14 07:15:00  70706.75  70706.75  70560.58  70684.01
231 2026-03-14 09:30:00  70702.81  70702.81  70542.58  70613.00
232 2026-03-14 09:45:00  70613.00  70613.00  70400.00  70501.16
255 2026-03-14 15:30:00  70684.05  70684.05  70501.00  70539.47
263 2026-03-14 17:30:00  70732.19  70732.19  70639.39  70663.54
277 2026-03-14 21:00:00  70805.71  70805.71  70661.06  70663.47
292 2026-03-15 00:45:00  71083.01  71083

In [2]:
import ccxt
import pandas as pd
import time

exchange = ccxt.binance()

symbol = 'BTC/USDT'
timeframe = '15m'

all_ohlcv = []
since = exchange.parse8601('2026-01-01T00:00:00Z')  # choose your start
limit = 1000

while True:
    batch = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since, limit=limit)
    
    if not batch:
        break

    all_ohlcv.extend(batch)
    since = batch[-1][0] + 1

    if len(batch) < limit:
        break

    time.sleep(exchange.rateLimit / 1000)

df = pd.DataFrame(all_ohlcv, columns=[
    'timestamp', 'open', 'high', 'low', 'close', 'volume'
])
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

print(df.head())
print(df.tail())
print("Total rows:", len(df))

            timestamp      open      high       low     close     volume
0 2026-01-01 00:00:00  87648.21  87816.00  87632.74  87742.61   86.09755
1 2026-01-01 00:15:00  87742.62  87776.19  87742.61  87776.18   24.11371
2 2026-01-01 00:30:00  87776.19  87776.19  87733.98  87748.89   51.85067
3 2026-01-01 00:45:00  87748.89  87849.26  87748.88  87809.23   71.59843
4 2026-01-01 01:00:00  87809.24  88013.00  87809.23  88000.00  104.72793
               timestamp      open      high       low     close     volume
7715 2026-03-22 08:45:00  68738.55  68825.00  68694.31  68804.77   67.25167
7716 2026-03-22 09:00:00  68804.77  68881.19  68750.94  68873.38   81.12297
7717 2026-03-22 09:15:00  68873.37  68901.65  68587.78  68649.09  177.61728
7718 2026-03-22 09:30:00  68649.09  68829.02  68644.58  68782.75   69.66217
7719 2026-03-22 09:45:00  68782.76  68782.76  68782.75  68782.75    0.00633
Total rows: 7720


In [4]:
import numpy as np

df['open_eq_high'] = np.isclose(df['open'], df['high'], atol=1e-6)

total = len(df)
signals = df['open_eq_high'].sum()

print("Total candles:", total)
print("Open == High signals:", signals)
print("Frequency:", signals / total)

Total candles: 7720
Open == High signals: 602
Frequency: 0.07797927461139896


In [5]:
#open = low
df['open_eq_low'] = np.isclose(df['open'], df['low'], atol=1e-6)

total = len(df)
signals = df['open_eq_low'].sum()
print("Open == Low signals:", signals)
print("Frequency:", signals / total)    
    

Open == Low signals: 539
Frequency: 0.06981865284974094


In [7]:
import ccxt
import pandas as pd
import numpy as np
import time

exchange = ccxt.binance()
symbol = 'BTC/USDT'
timeframe = '5m'

all_ohlcv = []
since = exchange.parse8601('2026-01-01T00:00:00Z')
limit = 1000

while True:
    batch = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since, limit=limit)
    if not batch:
        break
    all_ohlcv.extend(batch)
    since = batch[-1][0] + 1
    if len(batch) < limit:
        break
    time.sleep(exchange.rateLimit / 1000)

df5 = pd.DataFrame(all_ohlcv, columns=['timestamp','open','high','low','close','volume'])
df5['timestamp'] = pd.to_datetime(df5['timestamp'], unit='ms')
df5 = df5.sort_values('timestamp').reset_index(drop=True)

df5['bucket_15m'] = df5['timestamp'].dt.floor('15min')

rows = []

for bucket, g in df5.groupby('bucket_15m'):
    if len(g) != 3:
        continue

    g = g.sort_values('timestamp').reset_index(drop=True)

    open_15 = g.loc[0, 'open']
    first5_high = g.loc[0, 'high']
    first5_close = g.loc[0, 'close']

    full15_high = g['high'].max()
    full15_close = g.loc[2, 'close']

    cond = np.isclose(first5_high, open_15, atol=1e-6) and (first5_close < open_15)

    if cond:
        entry = first5_close
        rows.append({
            'bucket_15m': bucket,
            'entry': entry,
            'open_15': open_15,
            'full15_high': full15_high,
            'full15_close': full15_close,
            'covered_later': full15_high > open_15 + 1e-6,
            'close_above_open': full15_close > open_15,
            'ret_entry_to_close': full15_close / entry - 1
        })

test = pd.DataFrame(rows)

print("Setups:", len(test))
print("P(covered later):", test['covered_later'].mean())
print("P(close above open):", test['close_above_open'].mean())
print("Mean ret entry->close:", test['ret_entry_to_close'].mean())

Setups: 948
P(covered later): 0.5949367088607594
P(close above open): 0.2310126582278481
Mean ret entry->close: -5.260651637857403e-05
